Day-1

In [5]:

import ee
import geemap

# 1. Initialize Google Earth Engine
PROJECT_ID = "deforestation-detection-ml"
ee.Initialize(project=PROJECT_ID)

# 2. Define the study region: Western Ghats, Karnataka
# Study area: forested region near Kudremukh, Western Ghats, Karnataka
# Forest-focused study area near Kudremukh National Park
region = ee.Geometry.Rectangle([75.15, 13.10, 75.30, 13.25])

# 3. Load Sentinel-2 imagery for both years
def get_sentinel_collection(start_date, end_date):
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
    )

collection_2023 = get_sentinel_collection(
    "2023-01-01", "2024-01-01"
)
collection_2024 = get_sentinel_collection(
    "2024-01-01", "2025-01-01"
)

print("2023 images:", collection_2023.size().getInfo())
print("2024 images:", collection_2024.size().getInfo())

# 4. Calculate NDVI using near-infrared (B8) and red (B4)
def calculate_ndvi(image):
    return image.normalizedDifference(
        ["B8", "B4"]
    ).rename("NDVI")

ndvi_2023 = (
    collection_2023.map(calculate_ndvi)
    .median()
    .clip(region)
)

ndvi_2024 = (
    collection_2024.map(calculate_ndvi)
    .median()
    .clip(region)
)

# 5. Calculate NDVI change: 2024 minus 2023
ndvi_change = ndvi_2024.subtract(
    ndvi_2023
).rename("NDVI_Change")

# 6. Calculate regional change statistics
change_stats = ndvi_change.reduceRegion(
    reducer=ee.Reducer.mean().combine(
        reducer2=ee.Reducer.minMax(),
        sharedInputs=True
    ),
    geometry=region,
    scale=100,
    maxPixels=1e8,
    bestEffort=True
)

print("\nNDVI change statistics (2023–2024):")
print(change_stats.getInfo())

# 7. Visualize the results
Map = geemap.Map()
Map.centerObject(region, 10)

ndvi_vis = {
    "min": -0.2,
    "max": 0.8,
    "palette": ["brown", "yellow", "green"]
}

change_vis = {
    "min": -0.4,
    "max": 0.4,
    "palette": ["red", "white", "green"]
}

Map.addLayer(region, {}, "Study Region")
Map.addLayer(ndvi_2023, ndvi_vis, "NDVI 2023")
Map.addLayer(ndvi_2024, ndvi_vis, "NDVI 2024")
Map.addLayer(ndvi_change, change_vis, "NDVI Change")

Map

2023 images: 24
2024 images: 19

NDVI change statistics (2023–2024):
{'NDVI_Change_max': 0.3805156350135803, 'NDVI_Change_mean': 0.002196960511763713, 'NDVI_Change_min': -0.5007525980472565}


Map(center=[13.175003233269384, 75.22499999999984], controls=(WidgetControl(options=['position', 'transparent_…

Day-2

In [12]:

# ============================================================
# DAY 2 — SEASONAL VEGETATION CHANGE ANALYSIS
# Study area: Kudremukh region, Western Ghats, Karnataka
# ============================================================

import ee
import geemap

# 1. DEFINE THE SHARED STUDY AREA
region = ee.Geometry.Rectangle([75.15, 13.10, 75.30, 13.25])

# 2. MASK CLOUDS, SHADOWS, AND OTHER UNWANTED SCL CLASSES
def mask_sentinel2_clouds(image):
    scl = image.select("SCL")

    mask = (
        scl.neq(1)       # Saturated/defective pixels
        .And(scl.neq(3)) # Cloud shadows
        .And(scl.neq(8)) # Medium-probability clouds
        .And(scl.neq(9)) # High-probability clouds
        .And(scl.neq(10))# Cirrus
        .And(scl.neq(11))# Snow/ice
    )

    return image.updateMask(mask)


# 3. CALCULATE NDVI
def calculate_ndvi(image):
    return (
        image.normalizedDifference(["B8", "B4"])
        .rename("NDVI")
        .copyProperties(image, ["system:time_start"])
    )


# 4. CREATE SEASON-MATCHED NDVI COMPOSITES
def create_seasonal_ndvi(start_date, end_date):
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
        .map(mask_sentinel2_clouds)
    )

    count = collection.size().getInfo()
    print(f"Sentinel-2 {start_date} to {end_date}: {count} images")

    if count == 0:
        raise ValueError(f"No Sentinel-2 images found for {start_date}")

    return (
        collection.map(calculate_ndvi)
        .median()
        .clip(region)
    )


ndvi_2023 = create_seasonal_ndvi(
    "2023-03-01", "2023-05-01"
)

ndvi_2024 = create_seasonal_ndvi(
    "2024-03-01", "2024-05-01"
)

ndvi_change = ndvi_2024.subtract(ndvi_2023).rename("NDVI_Change")


# 5. SUMMARIZE NDVI CHANGE
summary_reducer = (
    ee.Reducer.mean()
    .combine(ee.Reducer.minMax(), sharedInputs=True)
)

ndvi_stats = ndvi_change.reduceRegion(
    reducer=summary_reducer,
    geometry=region,
    scale=100,
    maxPixels=1e8,
    bestEffort=True
)

print("\n=== NDVI CHANGE STATISTICS ===")
print(ndvi_stats.getInfo())


# 6. GET DYNAMIC WORLD TREE PROBABILITY
def get_tree_probability(start_date, end_date):
    collection = (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .select("trees")
    )

    count = collection.size().getInfo()
    print(f"Dynamic World {start_date} to {end_date}: {count} images")

    if count == 0:
        raise ValueError(f"No Dynamic World images found for {start_date}")

    return collection.median().clip(region)


trees_2023 = get_tree_probability(
    "2023-03-01", "2023-05-01"
)

trees_2024 = get_tree_probability(
    "2024-03-01", "2024-05-01"
)

tree_change = (
    trees_2024.subtract(trees_2023)
    .rename("Tree_Probability_Change")
)


# 7. SUMMARIZE TREE-PROBABILITY CHANGE
tree_stats = tree_change.reduceRegion(
    reducer=summary_reducer,
    geometry=region,
    scale=100,
    maxPixels=1e8,
    bestEffort=True
)

print("\n=== TREE PROBABILITY CHANGE ===")
print(tree_stats.getInfo())


# 8. IDENTIFY EXPLORATORY LOSS CANDIDATES
# Both indicators must decline by more than 0.10.
# These thresholds are exploratory, not validated labels.

ndvi_decline = ndvi_change.lt(-0.10)
tree_decline = tree_change.lt(-0.10)

loss_candidates = (
    ndvi_decline.And(tree_decline)
    .rename("Potential_Loss")
)

pixel_area_ha = ee.Image.pixelArea().divide(10000)

candidate_area = pixel_area_ha.updateMask(
    loss_candidates
).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=region,
    scale=10,
    maxPixels=1e8,
    bestEffort=True
)

print("\n=== EXPLORATORY LOSS CANDIDATES ===")
print("Candidate area in hectares:", candidate_area.getInfo())


# 9. DISPLAY THE FINAL INTERACTIVE MAP
Map_day2 = geemap.Map()
Map_day2.centerObject(region, 11)

Map_day2.addLayer(
    ndvi_2023,
    {"min": -0.2, "max": 0.8,
     "palette": ["brown", "white", "green"]},
    "NDVI 2023"
)

Map_day2.addLayer(
    ndvi_2024,
    {"min": -0.2, "max": 0.8,
     "palette": ["brown", "white", "green"]},
    "NDVI 2024"
)

Map_day2.addLayer(
    ndvi_change,
    {"min": -0.3, "max": 0.3,
     "palette": ["red", "white", "blue"]},
    "NDVI Change"
)

Map_day2.addLayer(
    trees_2023,
    {"min": 0, "max": 1,
     "palette": ["white", "lightgreen", "darkgreen"]},
    "Tree Probability 2023"
)

Map_day2.addLayer(
    trees_2024,
    {"min": 0, "max": 1,
     "palette": ["white", "lightgreen", "darkgreen"]},
    "Tree Probability 2024"
)

Map_day2.addLayer(
    tree_change,
    {"min": -0.3, "max": 0.3,
     "palette": ["red", "white", "green"]},
    "Tree Probability Change"
)

Map_day2.addLayer(
    loss_candidates.selfMask(),
    {"palette": ["red"]},
    "Potential Loss Candidates - Unvalidated"
)

Map_day2.addLayer(
    ee.Image().paint(region, 1, 2),
    {"palette": ["yellow"]},
    "Study Area Boundary"
)

Map_day2


Sentinel-2 2023-03-01 to 2023-05-01: 10 images
Sentinel-2 2024-03-01 to 2024-05-01: 11 images

=== NDVI CHANGE STATISTICS ===
{'NDVI_Change_max': 0.4152538329362869, 'NDVI_Change_mean': 0.01915146184855613, 'NDVI_Change_min': -0.43336284160614014}
Dynamic World 2023-03-01 to 2023-05-01: 10 images
Dynamic World 2024-03-01 to 2024-05-01: 7 images

=== TREE PROBABILITY CHANGE ===
{'Tree_Probability_Change_max': 0.4374673949787393, 'Tree_Probability_Change_mean': 0.018279043803680933, 'Tree_Probability_Change_min': -0.654054269194603}

=== EXPLORATORY LOSS CANDIDATES ===
Candidate area in hectares: {'area': 117.69950412735264}


Map(center=[13.175003233269384, 75.22499999999984], controls=(WidgetControl(options=['position', 'transparent_…